# 📖 Notebook 2: Denormalization Trade-offs

In Notebook 1, we built a clean, **normalized** schema — each fact stored exactly once. That's great for correctness, but what happens when reads get slow?

**Denormalization** means intentionally duplicating data to make reads faster at the cost of more complex writes.

## Learning Objectives

By the end of this notebook, you'll understand:
- What normalization gives you and what it costs
- When and why to denormalize
- How to measure the performance difference
- The consistency problems denormalization creates
- Real-world examples of denormalization in practice

## 🛠️ Setup

```bash
cd core-concepts/data-modeling
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "data_modeling_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.commit()
    conn.close()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 🧹 Normalized vs Denormalized: Side by Side

Our database has both versions of the posts data:

**Normalized** (`posts` + `users` + `likes` + `comments`):
- Each fact stored once — username in `users`, like count computed from `likes`
- Need JOINs to combine data
- Updates are simple (change username in one place)

**Denormalized** (`posts_denormalized`):
- Username, like_count, comment_count baked into every row
- No JOINs needed for reads
- Updates are complex (change username → update EVERY post row)

In [ ]:
# Compare the two approaches side by side

print("📋 NORMALIZED (separate tables, JOINs needed):")
print("=" * 70)
normalized = query("""
    SELECT p.id, u.username, p.content,
           COUNT(DISTINCT l.id) AS like_count,
           COUNT(DISTINCT c.id) AS comment_count
    FROM posts p
    JOIN users u ON p.user_id = u.id
    LEFT JOIN likes l ON l.post_id = p.id
    LEFT JOIN comments c ON c.post_id = p.id
    WHERE p.id <= 5
    GROUP BY p.id, u.username, p.content
    ORDER BY p.id;
""")
for row in normalized:
    print(f"  Post #{row['id']} by @{row['username']}: "
          f"{row['like_count']} likes, {row['comment_count']} comments")

print()
print("📋 DENORMALIZED (all in one table, no JOINs):")
print("=" * 70)
denormalized = query("""
    SELECT id, username, content, like_count, comment_count
    FROM posts_denormalized
    WHERE id <= 5
    ORDER BY id;
""")
for row in denormalized:
    print(f"  Post #{row['id']} by @{row['username']}: "
          f"{row['like_count']} likes, {row['comment_count']} comments")

print()
print("💡 Same data, different structure. The denormalized version is simpler to read")
print("   but harder to keep in sync when data changes.")

## ⏱️ Performance Comparison

The whole point of denormalization is **speed**. Let's measure it.

We'll compare two ways to load a feed of posts with like and comment counts:

In [ ]:
def benchmark(label, sql, iterations=100):
    """Run a query many times and report average latency."""
    times = []
    for _ in range(iterations):
        conn = get_db()
        cursor = conn.cursor()
        start = time.time()
        cursor.execute(sql)
        cursor.fetchall()
        elapsed = (time.time() - start) * 1000
        times.append(elapsed)
        conn.close()
    
    avg = sum(times) / len(times)
    p95 = sorted(times)[int(len(times) * 0.95)]
    return {"label": label, "avg": avg, "p95": p95, "min": min(times), "max": max(times)}

# Query 1: Normalized — requires JOINs and GROUP BY
normalized_result = benchmark(
    "Normalized (JOINs)",
    """
    SELECT p.id, u.username, p.content,
           COUNT(DISTINCT l.id) AS like_count,
           COUNT(DISTINCT c.id) AS comment_count
    FROM posts p
    JOIN users u ON p.user_id = u.id
    LEFT JOIN likes l ON l.post_id = p.id
    LEFT JOIN comments c ON c.post_id = p.id
    GROUP BY p.id, u.username, p.content
    ORDER BY p.created_at DESC
    LIMIT 20;
    """
)

# Query 2: Denormalized — simple SELECT, no JOINs
denormalized_result = benchmark(
    "Denormalized (no JOINs)",
    """
    SELECT id, username, content, like_count, comment_count
    FROM posts_denormalized
    ORDER BY created_at DESC
    LIMIT 20;
    """
)

print("⏱️ Performance Comparison (100 iterations each):")
print("=" * 60)
for r in [normalized_result, denormalized_result]:
    print(f"  {r['label']}:")
    print(f"    Avg: {r['avg']:.2f} ms | P95: {r['p95']:.2f} ms | Min: {r['min']:.2f} ms")
    print()

speedup = normalized_result['avg'] / denormalized_result['avg']
print(f"🚀 Denormalized is ~{speedup:.1f}× faster for reads!")
print()
print("💡 With 200 posts this difference is small. At millions of rows")
print("   with complex JOINs, denormalization can mean 10-100× improvement.")

## ⚠️ The Write Penalty

Denormalization makes reads faster, but writes become **more complex and error-prone**.

Consider what happens when a user changes their username:

In [ ]:
# Scenario: user_1 changes their username

print("📝 Scenario: user_1 changes username to 'alice_wonderland'")
print("=" * 60)

# Count how many rows we need to update in each approach
post_count = query("SELECT COUNT(*) as n FROM posts WHERE user_id = 1")[0]['n']
denorm_count = query("SELECT COUNT(*) as n FROM posts_denormalized WHERE user_id = 1")[0]['n']

print()
print("NORMALIZED approach:")
print(f"  UPDATE users SET username = 'alice_wonderland' WHERE id = 1;")
print(f"  → 1 row updated. Done! ✅")
print(f"  All {post_count} posts automatically show the new name via JOIN.")

print()
print("DENORMALIZED approach:")
print(f"  UPDATE users SET username = 'alice_wonderland' WHERE id = 1;")
print(f"  UPDATE posts_denormalized SET username = 'alice_wonderland' WHERE user_id = 1;")
print(f"  → 1 + {denorm_count} rows updated. ⚠️")
print(f"  If the second UPDATE fails, you have INCONSISTENT data!")

print()
print("💡 This is the core trade-off:")
print("   Normalized  = simple writes, complex reads")
print("   Denormalized = complex writes, simple reads")

In [ ]:
# Let's demonstrate the consistency problem

print("🐛 Simulating a consistency bug with denormalization:")
print("=" * 60)

# Add a new like to post #1 in the normalized tables
# but "forget" to update the denormalized table
conn = get_db()
cursor = conn.cursor()

# Check if user 50 already liked post 1
cursor.execute("SELECT COUNT(*) FROM likes WHERE user_id = 50 AND post_id = 1")
already_liked = cursor.fetchone()[0] > 0

if not already_liked:
    cursor.execute("INSERT INTO likes (user_id, post_id) VALUES (50, 1)")
    conn.commit()
conn.close()

# Now compare the two sources
real_likes = query(
    "SELECT COUNT(*) as n FROM likes WHERE post_id = 1"
)[0]['n']
cached_likes = query(
    "SELECT like_count FROM posts_denormalized WHERE id = 1"
)[0]['like_count']

print(f"  Normalized likes table:      {real_likes} likes on post #1")
print(f"  Denormalized like_count:      {cached_likes} likes on post #1")
if real_likes != cached_likes:
    print(f"  ❌ MISMATCH! Off by {real_likes - cached_likes}.")
    print(f"     The denormalized table is stale.")
else:
    print(f"  ✅ They match (for now — add another like and they'll diverge).")

print()
print("💡 In production, you'd use database triggers, background workers,")
print("   or event-driven updates to keep denormalized data in sync.")
print("   But they can still drift — that's the price of denormalization.")

## ✅ When to Denormalize

Denormalization makes sense in specific situations:

| Scenario | Why Denormalize? |
|----------|------------------|
| **Read-heavy systems** (1000:1 read/write) | Reads far outnumber writes — optimize for the common case |
| **Analytics / reporting** | Pre-aggregated data avoids expensive real-time JOINs |
| **Event logs / audit trails** | Capture snapshot at the time of the event |
| **Display names / labels** | Rarely change, frequently displayed |

**When NOT to denormalize:**

| Scenario | Why Not? |
|----------|----------|
| **Data changes frequently** | Too many places to update |
| **Strong consistency required** | Financial data, inventory counts |
| **Small dataset** | JOINs are fast enough — don't add complexity |

### The Cache Alternative

Often the best approach is: **keep your schema normalized, put a cache in front**.

Your database stays clean and correct. The cache holds denormalized, pre-computed data for fast reads. If the cache is wrong, you just rebuild it from the normalized source of truth.

In [ ]:
# Real-world comparison: Twitter timeline
# This is how companies think about the trade-off

print("🐦 Real-World Example: Twitter Timeline")
print("=" * 60)
print()
print("NORMALIZED approach (fan-out on read):")
print("  When you open Twitter, it runs:")
print("    SELECT posts FROM follows")
print("    JOIN posts ON posts.user_id = follows.following_id")
print("    WHERE follows.follower_id = you")
print("    ORDER BY created_at DESC LIMIT 20")
print("  ⚠️ If you follow 500 people, this JOIN is expensive.")
print()
print("DENORMALIZED approach (fan-out on write):")
print("  When someone posts, it writes to every follower's timeline:")
print("    For each follower: INSERT INTO timelines (user_id, post) ...")
print("  When you open Twitter: SELECT * FROM timelines WHERE user_id = you")
print("  ✅ Reads are instant! But writes are expensive.")
print()
print("Twitter's actual approach: HYBRID")
print("  - Regular users: fan-out on write (pre-compute timelines)")
print("  - Celebrities (10M+ followers): fan-out on read (too many writes)")
print()
print("💡 The best solution often combines both approaches!")

## 📚 Summary

### Key Takeaways

1. **Normalization** stores each fact once — great for consistency, harder for reads
2. **Denormalization** duplicates data for faster reads — but writes become complex
3. **Measure first** — don't denormalize until you've proven reads are too slow
4. **Consistency is the cost** — denormalized data can become stale or conflicting
5. **Cache first** — often a cache in front of normalized data is simpler than denormalizing

### Interview Tip

> Start with a normalized schema. If the interviewer pushes on read performance,  
> say: *"I'd denormalize the like count into the posts table since likes are read  
> far more often than written, and eventual consistency is acceptable here."*

### Next Up

In **Notebook 3**, we'll explore **NoSQL data models** — document stores, key-value patterns, and when relational isn't the right fit.